# Advanced Modeling With Integer Variables

In the last lesson, we saw that we can model complicated logical statements using integer constraints. As an example, we looked at  an infrastructure project selection model, where an agency is considering investing into several large infrastructure projects (Projects 1 to 5). $y_1, ... , y_5$ were binary variables - taking values of 1 if the corresponding project is selected and 0 if not.

The logic table below shows the different combinations of funding projects 1-3, and whether or not they satisify the selected constraint that dictates the relationship between the variables. With the help of the table, try selecting different constraints and reasoning through how the logic of the statement translates to the mathematical constraint.  

In [1]:
#@title Constraints Logic Table

import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd

# Define the possible values of y1, y2, y3
data = [
    {'y1': 0, 'y2': 0, 'y3': 0},
    {'y1': 0, 'y2': 0, 'y3': 1},
    {'y1': 0, 'y2': 1, 'y3': 0},
    {'y1': 0, 'y2': 1, 'y3': 1},
    {'y1': 1, 'y2': 0, 'y3': 0},
    {'y1': 1, 'y2': 0, 'y3': 1},
    {'y1': 1, 'y2': 1, 'y3': 0},
    {'y1': 1, 'y2': 1, 'y3': 1}
]
df = pd.DataFrame(data)

# Define constraints
constraints = {
    "y2 + y3 ≤ 1 + y1: If 2 and 3 selected, 1 must be selected": lambda row: row['y2'] + row['y3'] <= 1 + row['y1'],
    "y2 + y3 ≤ 2 - y1: 1 cannot be selected if both 2 and 3 are selected": lambda row: row['y2'] + row['y3'] <= 2 - row['y1'],
    "y1 + y2 + y3 = 2: exactly 2 of 1, 2 and 3 must be selected ": lambda row: row['y1'] + row['y2'] + row['y3'] == 2,
    "y1 ≤ y2: If 1 is selected, 2 must be selected": lambda row: row['y1'] <= row['y2'],
    "y1 ≤ y3: If 1 is selected, 3 must be selected": lambda row: row['y1'] <= row['y3'],
    "y1 ≤ y2 + y3 If 1 is selected, then either 2 or 3 must be selected": lambda row: row['y1'] <= row['y2'] + row['y3']  # New constraint
}

# Function to update the table based on the selected constraint
def update_table(constraint_name):
    constraint = constraints[constraint_name]
    df['Ok?'] = df.apply(constraint, axis=1)

    # Transpose the table
    transposed_data = df.T
    transposed_data.columns = [f"Case {i+1}" for i in range(len(df))]

    # Generate the HTML table with compact styling
    table_html = '''
    <style>
    table {
        border-collapse: collapse;
        width: 100%;
        font-family: Arial, sans-serif;
        border-radius: 8px;
        overflow: hidden;
        box-shadow: 0 2px 5px rgba(0, 0, 0, 0.2);
    }
    th, td {
        padding: 6px;  /* Reduced padding for tighter layout */
        text-align: center;
        font-size: 14px;  /* Slightly smaller font */
    }
    th {
        background-color: #4CAF50;
        color: white;
    }
    tr:nth-child(even) {
        background-color: #f2f2f2;
    }
    tr:hover {
        background-color: #ddd;
    }
    td {
        border-bottom: 1px solid #ddd;
    }
    .red {
        color: red;
        font-weight: bold;
    }
    </style>
    '''

    table_html += '<table>'
    table_html += '<tr><th></th>' + ''.join(f'<th>{col}</th>' for col in transposed_data.columns) + '</tr>'
    for index, row in transposed_data.iterrows():
        is_false_column = transposed_data.loc['Ok?'] == False if index != 'Ok?' else None
        table_html += '<tr>'
        # Subscript formatting for y indices
        header = f"<b>{index.replace('y', 'y<sub>')}</sub>" if 'y' in index else f"<b>{index}</b>"
        table_html += f'<td>{header}</td>'
        for i, val in enumerate(row):
            # Apply red color if the column fails the constraint, or if the "Ok?" value is False
            color_class = 'red' if (is_false_column is not None and is_false_column.iloc[i]) or (index == 'Ok?' and val == False) else ''
            cell_value = val if index != "Ok?" else ("Yes" if val else "No")
            table_html += f'<td class="{color_class}">{cell_value}</td>'
        table_html += '</tr>'
    table_html += '</table>'

    display(HTML(table_html))

# Create the radio buttons for constraint selection
radio_buttons = widgets.RadioButtons(
    options=list(constraints.keys()),
    description='Constraint:',
    value="y1 ≤ y2 + y3 If 1 is selected, then either 2 or 3 must be selected",
    style={'description_width': 'initial'},
    layout={'width': '800px'}  # Wider layout
)

# Add an interactive output to update the table based on the selection
output = widgets.Output()

def on_change(change):
    with output:
        output.clear_output()
        update_table(change['new'])

radio_buttons.observe(on_change, names='value')

# Display the widgets and initial table
display(radio_buttons)
with output:
    update_table(radio_buttons.value)
display(output)


RadioButtons(description='Constraint:', index=5, layout=Layout(width='800px'), options=('y2 + y3 ≤ 1 + y1: If …

Output()